In [ ]:
from collections import defaultdict
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from Costum_environments.LineWorld import LineWorld
from DRL_algorithms.Dynamic_methods import (
    iterative_policy_evaluation_sparse,
    policy_iteration_sparse,
    value_iteration_sparse
)

In [2]:
env = LineWorld(size=5, start_pos=2)

S = list(range(env.size))
A = ['left', 'right']
R = [-1.0, 0.0, 1.0]
terminal_states = [0, env.size - 1]

model = defaultdict(lambda: defaultdict(list))
valid_actions_dict = {}

for s in S:
    if s in terminal_states:
        continue
    for a in A:
        # Simule la transition
        next_s = s - 1 if a == 'left' else s + 1
        if next_s < 0 or next_s >= env.size:
            continue
        reward = -1.0 if next_s == 0 else (1.0 if next_s == env.size - 1 else 0.0)
        done = next_s in terminal_states
        model[s][a].append((1.0, next_s, reward, done))
        valid_actions_dict.setdefault(s, []).append(a)


In [3]:
# Politique aléatoire
pi_random = {
    s: {a: 1.0 / len(valid_actions_dict[s]) for a in valid_actions_dict[s]}
    for s in S if s not in terminal_states
}
V_random = iterative_policy_evaluation_sparse(
    pi=pi_random, S=S, A=A, model=model,
    terminal_states=terminal_states,
    valid_actions_dict=valid_actions_dict
)

# Politique gauche (toujours 'left')
pi_left = {
    s: {'left': 1.0} for s in S
    if s not in terminal_states and 'left' in valid_actions_dict[s]
}
V_left = iterative_policy_evaluation_sparse(
    pi=pi_left, S=S, A=A, model=model,
    terminal_states=terminal_states,
    valid_actions_dict=valid_actions_dict
)

# Politique droite (toujours 'right')
pi_right = {
    s: {'right': 1.0} for s in S
    if s not in terminal_states and 'right' in valid_actions_dict[s]
}
V_right = iterative_policy_evaluation_sparse(
    pi=pi_right, S=S, A=A, model=model,
    terminal_states=terminal_states,
    valid_actions_dict=valid_actions_dict
)

# Affichage comparé
print("=== Valeurs estimées sous 3 politiques différentes ===")
for s in S:
    v_r = V_random[s]
    v_l = V_left[s]
    v_d = V_right[s]
    print(f"État {s} : Aléatoire = {v_r:.3f} | Gauche = {v_l:.3f} | Droite = {v_d:.3f}")


=== Valeurs estimées sous 3 politiques différentes ===
État 0 : Aléatoire = 0.000 | Gauche = 0.000 | Droite = 0.000
État 1 : Aléatoire = -0.500 | Gauche = -1.000 | Droite = 0.980
État 2 : Aléatoire = -0.000 | Gauche = -0.990 | Droite = 0.990
État 3 : Aléatoire = 0.500 | Gauche = -0.980 | Droite = 1.000
État 4 : Aléatoire = 0.000 | Gauche = 0.000 | Droite = 0.000


In [4]:
pi_pi, V_pi = policy_iteration_sparse(
    S=S,
    A=A,
    R=R,
    model=model,
    terminal_states=terminal_states,
    valid_actions_dict=valid_actions_dict
)

print("=== Politique optimale et valeurs de V_pi (Policy Iteration) ===")
for s in S:
    if s in pi_pi:
        best_action = max(pi_pi[s], key=pi_pi[s].get)
        print(f"État {s} → action optimale : {best_action}, V[{s}] = {V_pi[s]:.3f}")
    else:
        print(f"État {s} (terminal) → V[{s}] = {V_pi[s]:.3f}")


=== Politique optimale et valeurs de V_pi (Policy Iteration) ===
État 0 (terminal) → V[0] = 0.000
État 1 → action optimale : right, V[1] = 0.980
État 2 → action optimale : right, V[2] = 0.990
État 3 → action optimale : right, V[3] = 1.000
État 4 (terminal) → V[4] = 0.000


In [5]:
pi_vi, V_vi = value_iteration_sparse(
    S=S,
    A=A,
    R=R,
    model=model,
    terminal_states=terminal_states,
    valid_actions_dict=valid_actions_dict
)

print("=== Politique optimale et valeurs de V_vi (Value Iteration) ===")
for s in S:
    if s in pi_vi:
        best_action = max(pi_vi[s], key=pi_vi[s].get)
        print(f"État {s} → action optimale : {best_action}, V[{s}] = {V_vi[s]:.3f}")
    else:
        print(f"État {s} (terminal) → V[{s}] = {V_vi[s]:.3f}")


=== Politique optimale et valeurs de V_vi (Value Iteration) ===
État 0 (terminal) → V[0] = 0.000
État 1 → action optimale : right, V[1] = 0.980
État 2 → action optimale : right, V[2] = 0.990
État 3 → action optimale : right, V[3] = 1.000
État 4 (terminal) → V[4] = 0.000
